# Case study 2: ecological functions of environmental genera

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaelmdc/braidworks/blob/main/notebooks/case-studies/microbial-functional-groups.ipynb)

**Question.** Which metabolic functional groups does FAPROTAX assign to ten genera
common in environmental samples, and on what evidence?

**Route.** Genus name → NCBI Taxonomy (taxon ID and lineage) → FAPROTAX (functional
groups, matched on the lineage). FAPROTAX's rule table is bundled with its source
package, so only the NCBI step uses the network (none at all when a local NCBI
taxonomy database is set up; the printed route shows which backend answered).

## Setup

Install Braidworks with every public source, then record the environment.

In [ ]:
%pip install -q "braidworks[pandas]" matplotlib

In [ ]:
import datetime
import importlib.metadata
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt

import braidworks as bw

OUT = Path("case-study-output")
OUT.mkdir(exist_ok=True)

# Figure style: one hue, muted axes, hairline grid (print-friendly, colour-blind safe).
INK, MUTED, GRID, ACCENT = "#0b0b0b", "#898781", "#e1e0d9", "#2a78d6"
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False,
})


def record_environment(sources):
    """Versions and date, saved next to the outputs so a run can be reproduced."""
    packages = ["braidworks-core", *sources]
    versions = {name: importlib.metadata.version(name) for name in packages}
    env = {
        "date_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        **versions,
    }
    for key, value in env.items():
        print(f"{key:24} {value}")
    return env


def timed(label, run):
    start = time.perf_counter()
    result = run()
    seconds = time.perf_counter() - start
    print(f"{label}: {seconds:.2f} s  ({result.summary()})")
    return result, seconds

In [ ]:
env = record_environment(['ncbi_weaver', 'faprotax_weaver'])

## The request

In [ ]:
GENERA = [
    "Methylococcus", "Nitrosomonas", "Nitrobacter", "Desulfovibrio", "Geobacter",
    "Methanosarcina", "Rhizobium", "Thiobacillus", "Pseudomonas", "Clostridium",
]
WANT = ["ncbi.taxon.id", "microbe.ecology.functional_groups", "microbe.ecology.evidence"]
print(bw.path("organism.name", WANT))

## Run it, cold and warm

In [ ]:
cache = OUT / "microbial-functional-groups-cache.sqlite"
cache.unlink(missing_ok=True)
with bw.Session(cache=cache) as session:
    result, cold_s = timed("cold", lambda: session.get(GENERA, have="organism.name", want=WANT))
with bw.Session(cache=cache) as session:
    _, warm_s = timed("warm", lambda: session.get(GENERA, have="organism.name", want=WANT))

In [ ]:
import pandas as pd

groups = {row["input"]: list(row["microbe.ecology.functional_groups"] or []) for row in result.rows}
functions = sorted({g for found in groups.values() for g in found})
matrix = pd.DataFrame(
    [[int(g in groups[genus]) for g in functions] for genus in GENERA],
    index=GENERA, columns=functions,
)
matrix.to_csv(OUT / "microbial-functional-groups.csv")
result.to_csv(OUT / "microbial-functional-groups-full.csv")
print({genus: len(found) for genus, found in groups.items()})

A genus with no group is not evidence of no function: FAPROTAX assigns groups only
where its curated literature rules match the lineage. The evidence column of the full
CSV lists the rule behind each assignment.

## Figure

In [ ]:
from matplotlib.colors import ListedColormap

shown = matrix.loc[:, matrix.sum() > 0].T  # one row per functional group
fig, ax = plt.subplots(figsize=(0.42 * shown.shape[1] + 2.6, 0.24 * shown.shape[0] + 1.2))
ax.imshow(shown.values, cmap=ListedColormap(["#f0efec", ACCENT]), aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(shown.shape[0]), [g.replace("_", " ") for g in shown.index], fontsize=7)
ax.set_xticks(range(shown.shape[1]), shown.columns, rotation=40, ha="right", fontsize=7,
              style="italic")
ax.set_xticks([x - 0.5 for x in range(1, shown.shape[1])], minor=True)
ax.set_yticks([y - 0.5 for y in range(1, shown.shape[0])], minor=True)
ax.grid(False)
ax.grid(which="minor", color="white", linewidth=1.5)
ax.tick_params(which="both", length=0)
ax.set_title("FAPROTAX functional groups assigned (filled) per genus", fontsize=8,
             color=MUTED, loc="left")
fig.tight_layout()
fig.savefig(OUT / "microbial-functional-groups.png")
fig.savefig(OUT / "microbial-functional-groups.svg")

In [ ]:
import json

env["timings_s"] = {"cold": round(cold_s, 2), "warm": round(warm_s, 2)}
(OUT / "microbial-functional-groups-environment.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env["timings_s"]))

## Sources to cite

In [ ]:
print(result.cite())

## Reproducing this case study

The sources are live databases, so counts change as they are updated: compare your
numbers with the environment record and date saved in `microbial-functional-groups-environment.json`, not
with the figures in a paper. Timings depend on the network and on provider load; the
warm run shows what a lookup cache saves when an analysis is rerun.